# Speaker verification with ESPnet-SPK

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/espnet/notebook/blob/master/Demos/spk_demo.ipynb) [![spk_demo](https://github.com/espnet/notebook/actions/workflows/spk_demo.yml/badge.svg)](https://github.com/espnet/notebook/actions/workflows/spk_demo.yml)

Turn a recording into one vector that describes the voice rather than the
words. Two recordings of the same person land close together; two people
land far apart.

CPU. Every ESPnet speaker model loads through the same call, so the
embedding is one line to change, and the end of the notebook compares them.

## Install


In [ ]:
%pip install -q "espnet[spk]==202610.post1" espnet_model_zoo librosa

# The WavLM models load their front end through S3PRL, which the 202610.post1
# spk extra does not pull in yet. Uncomment for those models:
# %pip install -q espnet-s3prl

## Two recordings

Two different sentences, read by two different people.


In [ ]:
import librosa
from IPython.display import Audio, display

!wget -q -O a.wav https://github.com/espnet/espnet/raw/master/test_utils/ctc_align_test.wav
!wget -q -O b.wav https://github.com/espnet/espnet/raw/master/test_utils/st_test.wav

for name in ("a.wav", "b.wav"):
    audio, rate = librosa.load(name, sr=16000)
    print(name)
    display(Audio(audio, rate=rate))


## An embedding each

Pick the model with `MODEL` — a drop-down in Colab, and any other ESPnet
speaker model id on the Hub works too. All are trained on VoxCeleb1+2 (the
`voxcelebs12devs_*` ones add more data) and released under CC BY 4.0. EER is
on VoxCeleb1-O, as each model card reports it; lower is better.

| Model | Encoder | Front end | EER (%) | Size |
|---|---|---|---|---|
| [`voxcelebs12_rawnet3`](https://huggingface.co/espnet/voxcelebs12_rawnet3) | RawNet3 | raw waveform | 0.739 | 114 MB |
| [`voxcelebs12_ecapa_mel`](https://huggingface.co/espnet/voxcelebs12_ecapa_mel) | ECAPA-TDNN | mel | 0.856 | 109 MB |
| [`voxcelebs12_ska_mel`](https://huggingface.co/espnet/voxcelebs12_ska_mel) | SKA-TDNN | mel | 0.729 | 193 MB |
| [`voxcelebs12_mfaconformer_mel`](https://huggingface.co/espnet/voxcelebs12_mfaconformer_mel) | MFA-Conformer | mel | 0.782 | 261 MB |
| [`voxcelebs12_xvector_mel`](https://huggingface.co/espnet/voxcelebs12_xvector_mel) | x-vector | mel | 1.814 | 151 MB |
| [`voxcelebs12_ecapa_frozen`](https://huggingface.co/espnet/voxcelebs12_ecapa_frozen) | ECAPA-TDNN | WavLM, frozen | 0.638 | 1.4 GB |
| [`voxcelebs12_ecapa_wavlm_joint`](https://huggingface.co/espnet/voxcelebs12_ecapa_wavlm_joint) | ECAPA-TDNN | WavLM, fine-tuned | 0.394 | 1.4 GB |
| [`voxcelebs12_ska_wavlm_frozen`](https://huggingface.co/espnet/voxcelebs12_ska_wavlm_frozen) | SKA-TDNN | WavLM, frozen | 0.564 | 2.1 GB |
| [`voxcelebs12_ska_wavlm_joint`](https://huggingface.co/espnet/voxcelebs12_ska_wavlm_joint) | SKA-TDNN | WavLM, fine-tuned | 0.516 | 2.1 GB |
| [`voxcelebs12devs_voxblinkclean_rawnet3`](https://huggingface.co/espnet/voxcelebs12devs_voxblinkclean_rawnet3) | RawNet3 | raw waveform, + VoxBlink clean | 0.771 | 241 MB |
| [`voxcelebs12devs_voxblinkfull_rawnet3`](https://huggingface.co/espnet/voxcelebs12devs_voxblinkfull_rawnet3) | RawNet3 | raw waveform, + VoxBlink full | 0.782 | 377 MB |
| [`voxcelebs12devs_librispeech_cv16fa_rawnet3`](https://huggingface.co/espnet/voxcelebs12devs_librispeech_cv16fa_rawnet3) | RawNet3 | raw waveform, + LibriSpeech, CV16 | 1.229 | 151 MB |

The WavLM rows need the commented install line above and are much faster on a
GPU. `load` keeps every model it has built, so switching back costs nothing.

In [ ]:
import numpy as np
import torch
from espnet2.bin.spk_inference import Speech2Embedding

MODEL = "espnet/voxcelebs12_rawnet3"  # @param ["espnet/voxcelebs12_rawnet3", "espnet/voxcelebs12_ecapa_mel", "espnet/voxcelebs12_ska_mel", "espnet/voxcelebs12_mfaconformer_mel", "espnet/voxcelebs12_xvector_mel", "espnet/voxcelebs12_ecapa_frozen", "espnet/voxcelebs12_ecapa_wavlm_joint", "espnet/voxcelebs12_ska_wavlm_frozen", "espnet/voxcelebs12_ska_wavlm_joint", "espnet/voxcelebs12devs_voxblinkclean_rawnet3", "espnet/voxcelebs12devs_voxblinkfull_rawnet3", "espnet/voxcelebs12devs_librispeech_cv16fa_rawnet3"] {allow-input: true}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
_models = {}


def load(model_id):
    if model_id not in _models:
        _models[model_id] = Speech2Embedding.from_pretrained(model_id, device=DEVICE)
    return _models[model_id]


def embed(audio, model_id=None):
    """One vector for a path or a 16 kHz array."""
    if isinstance(audio, str):
        audio, _ = librosa.load(audio, sr=16000)
    with torch.no_grad():
        return load(model_id or MODEL)(audio)[0].cpu().numpy()


first, second = embed("a.wav"), embed("b.wav")
print(MODEL, "embedding size:", first.shape)

## Same speaker or not

The score is the cosine between the two vectors. Where to put the line is a
choice about which mistakes you would rather make, not something the model
records — so it is a number you pick, and 0.36 is only a place to start.


In [ ]:
def cosine(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

# the same recording, cut in two: the same voice either way
audio, _ = librosa.load("a.wav", sr=16000)
half = len(audio) // 2
same = cosine(embed(audio[:half]), embed(audio[half:]))
different = cosine(first, second)

print(f"same speaker       {same:.3f}")
print(f"different speakers {different:.3f}")

## Model against model

The same test on several models. The list is the ones that run comfortably on
CPU; add the WavLM ones when you have a GPU. A cosine from one model is not
comparable with a cosine from another, so read the gap between the two
columns, and pick a threshold per model on recordings like yours.

In [ ]:
COMPARE = [
    "espnet/voxcelebs12_rawnet3",
    "espnet/voxcelebs12_ecapa_mel",
    "espnet/voxcelebs12_ska_mel",
    "espnet/voxcelebs12_xvector_mel",
    # "espnet/voxcelebs12_ecapa_wavlm_joint",  # the lowest EER; use a GPU
]

other, _ = librosa.load("b.wav", sr=16000)
print(f"{'model':40s} {'dim':>4s} {'same':>6s} {'diff':>6s} {'gap':>6s}")
for model_id in COMPARE:
    s = cosine(embed(audio[:half], model_id), embed(audio[half:], model_id))
    d = cosine(embed(audio, model_id), embed(other, model_id))
    dim = embed(other, model_id).shape[0]
    print(f"{model_id:40s} {dim:4d} {s:6.3f} {d:6.3f} {s - d:6.3f}")

## Where next

- **In the browser**: the [speaker-verification Space](https://huggingface.co/spaces/espnet/speaker-verification) does this with
  two files you drop in
- **Other checkpoints**: every speaker model in the
  [espnet organization](https://huggingface.co/espnet?search=voxceleb) loads
  the same way
- **Diarization** — who spoke when, built on these embeddings:
  [`../Courses/CMUSpeechTechnology26S/speaker_verification.ipynb`](../Courses/CMUSpeechTechnology26S/speaker_verification.ipynb)
